In [20]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

import operator


In [5]:
load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini")

In [6]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback on the essay, including strengths and areas for improvement.")
    score: float = Field(description="A score between 0 and 10 evaluating the overall quality of the essay.", ge = 0, le = 10)

In [7]:
Structured_model = model.with_structured_output(EvaluationSchema)

In [17]:
essay = """### **From Neural Circuits to Intelligent Systems: Advancing the Synergy Between Neuroscience and Artificial Intelligence**

The convergence of neuroscience and artificial intelligence (AI) represents one of the most transformative intellectual partnerships of the 21st century. Neuroscience seeks to unravel the mechanisms of the human brain—arguably the most complex system known—while AI attempts to replicate aspects of intelligence in machines. What began as a relationship of inspiration has matured into a dynamic, bidirectional exchange: neuroscience informs the design of intelligent systems, and AI, in turn, accelerates discoveries about the brain itself.

This connection is deeply rooted in biology. Early artificial neural networks were loosely inspired by neurons, but modern AI has drawn from far more precise neuroscientific findings. A notable example is the work of David Hubel and Torsten Wiesel, whose studies on the visual cortex revealed hierarchical feature detection—edges, shapes, and complex patterns. These discoveries directly influenced the development of convolutional neural networks (CNNs), now foundational in computer vision. Similarly, attention mechanisms in modern AI models echo cognitive processes studied in neuroscience, where selective focus allows the brain to prioritize relevant stimuli. These parallels demonstrate that AI is not merely inspired by the brain—it increasingly reflects its functional principles.

Learning mechanisms further illustrate this synergy. In neuroscience, synaptic plasticity governs how connections between neurons strengthen or weaken over time. Hebbian theory—often summarized as “cells that fire together wire together”—has conceptual parallels in machine learning. Reinforcement learning, a dominant paradigm in AI, is strongly influenced by the brain’s reward systems, particularly dopamine signaling. Empirical studies on reward prediction errors have shaped algorithms that enable AI agents to learn from feedback. This connection is evident in systems that achieve superhuman performance in complex environments, where learning emerges from iterative interaction rather than explicit instruction.

While neuroscience has shaped AI, the reverse influence is equally profound—and increasingly indispensable. The sheer scale of neural data presents a major analytical challenge, one that AI is uniquely suited to address. Machine learning models now play a central role in interpreting brain imaging data, uncovering patterns that were previously inaccessible. For example, AI systems have been used to reconstruct approximate visual images from fMRI signals, effectively “decoding” what a person is seeing. In another case, deep learning models have identified early biomarkers of Alzheimer’s disease by detecting subtle structural changes in brain scans years before clinical symptoms appear. These case studies highlight how AI is transforming neuroscience from a descriptive science into a predictive one.

The integration becomes even more tangible in the field of brain-computer interfaces (BCIs). These systems use AI to translate neural activity into actionable outputs, enabling direct communication between the brain and external devices. Recent breakthroughs have demonstrated the ability of AI-driven BCIs to restore movement in paralyzed individuals and enable text generation directly from neural signals. Such advancements are not merely technological milestones—they redefine the boundaries of human capability. At the same time, they exemplify the seamless fusion of biological and artificial intelligence.

However, this progress is accompanied by significant challenges. Despite their capabilities, AI systems remain fundamentally different from the human brain. They require vast datasets, lack true general intelligence, and struggle with adaptability in unfamiliar contexts. Neuroscience, too, is incomplete; core questions about consciousness, subjective experience, and higher-order reasoning remain unresolved. These gaps highlight the limits of current approaches and underscore the need for deeper integration between the fields.

Ethical considerations add another layer of complexity. The use of neural data raises critical concerns about privacy, consent, and ownership. If brain signals can be decoded, who controls that information? Moreover, as BCIs and neuro-AI systems become more advanced, questions about autonomy and identity emerge. Addressing these issues requires more than acknowledgment—it demands structured solutions. Emerging frameworks emphasize principles such as “neuro-rights,” which advocate for mental privacy, cognitive liberty, and protection against algorithmic manipulation. Regulatory models, interdisciplinary oversight, and transparent AI systems will be essential to ensure that innovation does not outpace ethical responsibility.

Looking forward, the future of this convergence lies in both technological and conceptual breakthroughs. Neuromorphic computing, which seeks to replicate the brain’s architecture in hardware, offers a path toward more efficient and adaptive AI systems. At the same time, collaborative research integrating neuroscience, computer science, and cognitive psychology will be crucial for advancing our understanding of intelligence itself. Rather than treating the brain as a system to imitate, future efforts may focus on co-evolution—designing AI that complements and extends human cognition.


In conclusion, neuroscience and artificial intelligence are engaged in a powerful and evolving dialogue. Neuroscience provides the empirical foundation, revealing how intelligence emerges from biological systems, while AI offers the tools to model, test, and expand those insights. Their intersection is not merely a point of overlap but a fertile ground for innovation. As this relationship deepens, it holds the promise of not only advancing technology but also unlocking a more profound understanding of the human mind—bridging the gap between natural and artificial intelligence in ways once thought impossible.
"""

In [ ]:
response = Structured_model.invoke(essay)

response

EvaluationSchema(feedback='The essay provides a comprehensive overview of the relationship between neuroscience and artificial intelligence, highlighting key areas of intersection and mutual influence. It successfully explains complex concepts in a clear and engaging manner, making it accessible to a broad audience. Strengths include the thorough analysis of how neuroscience informs AI design through examples like CNNs and reinforcement learning, as well as the reverse influence of AI on neuroscientific research. The discussion of ethical considerations and future directions adds depth and acknowledges the complexities involved in this evolving field.\n\nHowever, areas for improvement include:\n1. **Depth of Ethical Discussion**: While the essay touches on ethical issues, it could benefit from a more detailed examination of specific ethical dilemmas and potential recommendations for addressing them.\n2. **More Examples**: Additional case studies or examples of successful AI application

In [22]:
class EssayState(TypedDict):
    essay : str
    language_feedback : str
    depth_of_analysis_feedback : str
    clarity_of_though_feedback : str
    overall_feedback : str
    individual_scores : Annotated[list[float], operator.add]
    avg_score: float


In [23]:
def evaluate_language(state: EssayState):
     response = Structured_model.invoke(state['essay'])

     return {
        'language_feedback': response.feedback,
        'individual_scores': [response.score]
     }

In [ ]:
def evaluate_depth_of_analysis(state: EssayState) -> EssayState:
    response = Structured_model.invoke(state['essay'])
    state['depth_of_analysis_feedback'] = response.feedback
    state['individual_scores'].append(response.score)
    return state

In [ ]:
#create the graph
graph = StateGraph(EssayState)

#create nodes
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_depth_of_analysis', evaluate_depth_of_analysis)
graph.add_node('evaluate_clarity_of_thought', evaluate_clarity_of_thought)
graph.add_node('final_evaluation', final_evaluation)

In [ ]:
#nodes
